# Milestone 1 - Problem Framing and Data Acquisition
**Project:** Customer and Sales Intelligence for *Mavuno Mart*, a (fictional) growing Kenyan retail chain

## 1. The business problem
Mavuno Mart runs 20 stores in 11 counties. Management has till data but little insight. They want to know:

| # | Business question | Owner | Decision it informs |
|---|---|---|---|
| Q1 | Which **stores, counties and categories** drive revenue and growth? | Ops Director | Where to invest staff, space and stock |
| Q2 | **When** do customers buy - weekdays, paydays, holidays, festive and school seasons? | Supply chain | Staffing rosters and replenishment timing |
| Q3 | **Who** buys what - which customer segments matter most, and how do they pay (M-Pesa vs cash vs card)? | Marketing | Loyalty offers, payment partnerships |
| Q4 | **Did our promotions work?** (especially the March 2026 cooking-oil store test) | Commercial | Whether to repeat or stop a promotion |
| Q5 | Which **customers are drifting away**, and can we spot them early? | CRM | Who to target with win-back offers |
| Q6 | What **demand should we expect next month** by store and category? | Supply chain | Purchase orders for October 2026 |
| Q7 | How much of our sales "growth" is just **inflation**? | Finance | Honest performance reporting |

## 2. Success measures (agreed before looking at results)
* **Descriptive (Q1-Q3, Q7):** every headline number is reproducible from SQL and pandas and reconciles within 0.1%.
* **Promotion test (Q4):** a pre-registered hypothesis test at alpha = 0.05 with a 95% confidence interval for the uplift.
* **Demand forecast (Q6):** on a 3-month hold-out (Jul-Sep 2026), WAPE at store x category x day is **at least 15% better
  than a seasonal-naive baseline**, and monthly chain-level category forecasts are within +/-10%.
* **Churn model (Q5):** ROC-AUC >= 0.75 on a later, out-of-time cohort and the top-risk decile captures **>= 2.5x** the
  average churn rate (lift), so a win-back budget aimed at it is 2.5x more efficient than random targeting.

## 3. Data needed and how it was obtained
| Dataset | Content | Source / how obtained | Licence / terms |
|---|---|---|---|
| `stores.csv` | 20 stores: county, town, format, size, open date | Synthetic, `src/generate_data.py` | Own work (MIT, same as repo) |
| `products.csv` | 91 SKUs, 9 categories, KES price points (Oct 2024) | Synthetic; prices set to typical Nairobi supermarket shelf prices | Own work |
| `customers.csv` | 6,000 loyalty members: gender, birth year, signup, home store | Synthetic | Own work |
| `transactions.csv` / `transaction_items.csv` | 328k receipts, 928k lines, Oct 2024 - Sep 2026, payment method incl. M-Pesa | Synthetic | Own work |
| `promotions.csv` | 9 promotions incl. two randomised store tests | Synthetic | Own work |
| `kenya_inflation_monthly.csv` | 12-month and annual-average CPI inflation, Jan 2024 - Sep 2026 | KNBS CPI as published by the Central Bank of Kenya, centralbank.go.ke/inflation-rates (copied Oct 2026) | Public government statistics; no licence stated - reuse with attribution to KNBS/CBK |
| `county_population_2019.csv` | 2019 census population for the 11 store counties plus Meru (expansion candidate) | KNBS 2019 Kenya Population and Housing Census, Vol. I (via Wikipedia "List of counties of Kenya by population") | KNBS public data; Wikipedia text CC BY-SA 4.0 |
| `kenya_public_holidays.csv` | Gazetted public holidays Oct 2024 - Dec 2026 incl. Sunday-substitution days | Public Holidays Act (Cap. 110) + Gazette/Interior Ministry declarations for Idd-ul-Fitr (31 Mar 2025, 20 Mar 2026) | Public law - no restriction |

**Why synthetic sales data?** Real retailer till data is commercially confidential, and the brief allows a generated dataset.
The generator is deterministic (seed 2026) and *realistic by design*: real inflation drives prices, real holidays shift demand,
M-Pesa dominates payments, customers churn, and it contains real-world data-quality problems. It also plants some known
business events (a competitor opening near Nakuru in Feb 2026; one promotion that works and one that does not) so we can
check whether the analysis rediscovers them. The planted truth lives in `data/raw/_ground_truth_customers.csv` and is
**never** used for modelling.

In [1]:
import sys; sys.path.append("../src")   # lets the notebook import the project modules
import pandas as pd
from config import RAW, EXTERNAL

files = {p.name: p for p in sorted(RAW.glob("*.csv")) if not p.name.startswith("_")}
files.update({p.name: p for p in sorted(EXTERNAL.glob("*.csv"))})
inventory = []
for name, p in files.items():
    df = pd.read_csv(p, low_memory=False)
    inventory.append({"file": name, "rows": len(df), "columns": df.shape[1],
                      "missing_cells_%": round(100 * df.isna().mean().mean(), 2),
                      "size_MB": round(p.stat().st_size / 1e6, 2)})
pd.DataFrame(inventory)

,file,rows,columns,missing_cells_%,size_MB
0,customers.csv,6030,5,0.60,0.18
1,products.csv,91,6,0.73,0.01
2,promotions.csv,9,8,0.00,0.00
3,stores.csv,20,7,0.00,0.00
4,transaction_items.csv,964203,7,13.95,28.84
5,transactions.csv,329599,7,7.30,17.36
6,county_population_2019.csv,12,2,0.00,0.00
7,kenya_inflation_monthly.csv,33,4,0.00,0.00
8,kenya_public_holidays.csv,31,3,0.00,0.00


## 4. First look at the raw tables
The raw files are exactly what a POS export looks like - including the problems. A quick peek:

In [2]:
tx = pd.read_csv(RAW / "transactions.csv", dtype={"customer_id": str})
tx.head(8)

,transaction_id,transaction_datetime,store_id,till_no,customer_id,payment_method,total_amount
0,T0087926,2025-04-16 11:42:32,S012,8,NaN,Card,280.0
1,T0094657,2025-05-01 15:00:24,S008,3,C01387,M-Pesa,KES 470.00
2,T0207212,2025-12-31 13:26:55,S011,2,C05513,Cash,1080.0
3,T0213762,2026-01-16 17:19:41,S007,1,NaN,M-Pesa,430.0
4,T0062302,2025-02-14 17:38:02,S019,7,NaN,M-Pesa,513.0
5,T0119611,2025-06-29 17:14:40,S007,4,NaN,M-Pesa,196.0
6,T0178310,2025-11-07 18:06:12,S013,3,C00661,Cash,745.0
7,T0000170,2024-10-01 18:35:09,S016,5,C00884,M-Pesa,230.0


In [3]:
print("Payment method labels as typed at the till:")
print(tx.payment_method.value_counts(dropna=False).to_string())

Payment method labels as typed at the till:
payment_method
M-Pesa             177098
Cash                79601
Card                41814
Airtel Money         8829
NaN                  2637
mpesa                2303
MPESA                2293
Mpesa                2289
M-PESA               2254
M PESA               2214
Csh                  1717
cash                 1621
CASH                 1587
card                  920
Visa/Mastercard       917
CARD                  890
Airtel                317
AIRTEL MONEY          298


In [4]:
items = pd.read_csv(RAW / "transaction_items.csv", dtype={"promo_id": str})
items.describe().round(2)

,line_no,quantity,unit_price,discount_amount
count,964203.00,964203.00,958418.00,964203.00
mean,2.61,1.40,235.71,0.84
std,4.98,5.75,229.11,7.47
min,1.00,-7.00,25.00,-117.00
25%,1.00,1.00,95.00,0.00
50%,2.00,1.00,185.00,0.00
75%,3.00,2.00,270.00,0.00
max,91.00,1000.00,2620.00,900.00


Already visible: payment methods typed 15+ ways, `KES 1,234.00` strings in `total_amount`, missing prices, a quantity of
1000 for a single SKU, and duplicate rows. These are handled in Milestone 2.

## 5. Data model
A classic retail **star schema**: one fact table at receipt-line grain, with dimensions for store, product, customer,
date and promotion; context data join on `county` (population) and `month` (inflation) or `date` (holidays).

```
            dim_customer          dim_product
                  \                  /
 dim_store --- fact_sales_line (transaction_id, line_no) --- dim_date --- holidays / CPI
                  /
           dim_promotion          county population -> dim_store.county
```

In [5]:
pd.read_csv(RAW / "stores.csv")

,store_id,store_name,county,town,store_format,size_sqm,open_date
0,S001,Mavuno Mart Westlands,Nairobi,Westlands,Hypermarket,4200,2016-03-01
1,S002,Mavuno Mart CBD Moi Avenue,Nairobi,Nairobi CBD,Supermarket,1800,2014-06-01
2,S003,Mavuno Mart Embakasi,NAIROBI,Embakasi,Supermarket,2100,2018-09-01
3,S004,Mavuno Mart Kasarani,Nairobi,Kasarani,Supermarket,1900,2019-02-01
4,S005,Mavuno Mart Karen,Nairobi,Karen,Express,650,2021-05-01
5,S006,Mavuno Mart Rongai,Nairobi,Rongai,Express,700,2022-01-15
6,S007,Mavuno Mart Nyali,Mombasa,Nyali,Hypermarket,3900,2017-11-01
7,S008,Mavuno Mart Mombasa Town,Mombasa,Mombasa CBD,Supermarket,1600,2015-04-01
8,S009,Mavuno Mart Thika,Kiambu,Thika,Supermarket,2000,2018-03-01
9,S010,Mavuno Mart Ruiru,Kiambu,Ruiru,Express,800,2021-09-01


In [6]:
pd.read_csv(RAW / "promotions.csv")

,promo_id,promo_name,start_date,end_date,discount_pct,scope_type,scope_value,store_ids
0,PR001,Festive Drinks & Snacks 2024,2024-12-09,2024-12-31,10,category,Beverages,ALL
1,PR002,Back-to-School 2025,2024-12-30,2025-01-19,10,category,Stationery & School,ALL
2,PR003,Easter Dairy & Bakery,2025-04-14,2025-04-21,8,subcategory,Milk|Bakery,ALL
3,PR004,Personal Care Week,2025-07-07,2025-07-20,5,category,Personal Care,S002|S003|S004|S010|S011|S012|S014|S016|S017
4,PR005,Mid-Month Unga Saver,2025-11-14,2025-11-20,8,subcategory,Flour,ALL
5,PR006,Festive Drinks & Snacks 2025,2025-12-08,2025-12-31,10,category,Beverages,ALL
6,PR007,Back-to-School 2026,2025-12-29,2026-01-18,10,category,Stationery & School,ALL
7,PR008,Cooking Oil Price Drop (store test),2026-03-02,2026-03-22,15,subcategory,Cooking Oil,S001|S002|S004|S005|S009|S010|S013|S015|S016|S018
8,PR009,Household Cleaning Bonanza,2026-06-08,2026-06-21,7,subcategory,Cleaning,ALL
